# 21 — Four-Stage Refinement (`autocalibrate_four_stage`)

The v1 paper (paper3) defines calibration as **four stages**, and
`midas_calibrate_v2` reproduces that workflow with
`autocalibrate_four_stage`:

| stage | what it refines | why |
|---|---|---|
| **1 — geom only** | Lsd, BC, tilts; distortion + panels frozen | stable warm-start |
| **2 — full** | + distortion harmonics (+ panels) | analytical distortion |
| **3 — spline** | thin-plate spline on the residual ΔR map | localised distortion the basis can't represent |
| **4 — eval** | re-evaluate strain with stages 2+3 applied | honest final floor |

Stage 3 fits a TPS (`scipy.RBFInterpolator`) to the per-fit ΔR
residuals on the cake, with a held-out **test** set so the reported
floor isn't an over-fit.

This notebook is **self-contained synthetic** — it renders a CeO2 ring
image and runs all four stages.  Runtime ~15 s on a CPU.


In [1]:
import os, time
os.environ.setdefault('KMP_DUPLICATE_LIB_OK', 'TRUE')   # macOS OpenMP guard
import numpy as np
import torch

from midas_integrate.geometry import build_tilt_matrix, pixel_to_REta
from midas_calibrate import CalibrationParams, build_ring_table


def make_truth(Lsd_um=1_000_000.0) -> CalibrationParams:
    """Known-truth CeO2 geometry on a small 1024x1024 detector."""
    p = CalibrationParams()
    p.NrPixelsY = 1024; p.NrPixelsZ = 1024
    p.pxY = 200.0; p.pxZ = 200.0
    p.Lsd = Lsd_um
    p.BC_y = 512.0; p.BC_z = 512.0
    p.tx = 0.0; p.ty = 0.4; p.tz = 0.25
    p.Wavelength = 0.173
    p.SpaceGroup = 225
    p.LatticeConstant = (5.411, 5.411, 5.411, 90.0, 90.0, 90.0)
    p.MaxRingRad = 480.0
    p.MinRingRad = 0.0
    p.RhoD = 512.0
    p.Width = 1500.0
    p.EtaBinSize = 10.0
    p.RBinSize = 1.0
    p.nIterations = 4
    p.RemoveOutliersBetweenIters = False
    p.SNRMin = 1.5
    p.tolLsd = 5000.0; p.tolBC = 8.0; p.tolTilts = 1.0
    p.tolDistortion = 0.0
    p.Refine = {
        'Lsd': True, 'BC': True, 'ty': True, 'tz': True,
        'Wavelength': False, 'Parallax': False,
        **{f'p{i}': False for i in range(15)},
    }
    return p


def simulate_image(params, ring_thickness_px: float = 1.5) -> np.ndarray:
    """Render a 2D image: bright Gaussian rings on a noisy background."""
    rt = build_ring_table(params)
    NY, NZ = params.NrPixelsY, params.NrPixelsZ
    px = 0.5 * (params.pxY + params.pxZ)
    TRs = build_tilt_matrix(params.tx, params.ty, params.tz)
    Y_grid, Z_grid = np.meshgrid(np.arange(NY, dtype=np.float64),
                                 np.arange(NZ, dtype=np.float64))
    R_pix, _ = pixel_to_REta(
        Y_grid, Z_grid, Ycen=params.BC_y, Zcen=params.BC_z, TRs=TRs,
        Lsd=params.Lsd, RhoD=params.RhoD, px=px, parallax=params.Parallax)
    img = np.full(R_pix.shape, 50.0, dtype=np.float64)
    rng = np.random.default_rng(0)
    img += rng.normal(0, 5.0, size=img.shape)
    for r_ideal in rt.r_ideal_px:
        I_amp = 1000.0 / (1.0 + r_ideal / 100.0)
        img += I_amp * np.exp(-0.5 * ((R_pix - r_ideal) / ring_thickness_px) ** 2)
    return img


## Render a synthetic CeO2 image and build a perturbed seed


In [2]:
truth = make_truth()
image = simulate_image(truth)

seed = make_truth()
seed.Lsd += 300.0; seed.BC_y += 1.5; seed.BC_z -= 1.0
seed.ty -= 0.05; seed.tz += 0.06
print(f'image {image.shape}; truth Lsd={truth.Lsd:.0f} '
      f'BC=({truth.BC_y},{truth.BC_z}) ty={truth.ty} tz={truth.tz}')


image (1024, 1024); truth Lsd=1000000 BC=(512.0,512.0) ty=0.4 tz=0.25


## Run all four stages

`autocalibrate_four_stage(v1_params, image)` returns a
`FourStageResult` carrying each stage's geometry + strain, the trained
spline callable, and the stage-4 strains (full-set and held-out test).


In [3]:
from midas_calibrate_v2.pipelines.four_stage import autocalibrate_four_stage

t0 = time.time()
res = autocalibrate_four_stage(
    seed, image,
    n_iter_stage1=2, n_iter_stage2=2,
    enable_stage3_spline=True,
    spline_holdout_frac=0.2,
    verbose=False,
)
print(f'four-stage done in {time.time()-t0:.1f}s')


DIPlib -- a quantitative image analysis library
Version 3.5.2 (Dec 27 2024)
For more information see https://diplib.org


/Users/hsharma/miniconda3/envs/midas_env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


OMP: Info #276: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.


/Users/hsharma/opt/MIDAS/packages/midas_integrate/midas_integrate/kernels.py:396: UserWarning: Sparse invariant checks are implicitly disabled. Memory errors (e.g. SEGFAULT) will occur when operating on a sparse tensor which violates the invariants, but checks incur performance overhead. To silence this warning, explicitly opt in or out. See `torch.sparse.check_sparse_tensor_invariants.__doc__` for guidance.  (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/aten/src/ATen/Context.cpp:767.)
  return torch.sparse_csr_tensor(indptr, indices, values,
/Users/hsharma/opt/MIDAS/packages/midas_integrate/midas_integrate/kernels.py:396: UserWarning: Sparse CSR tensor support is in beta state. If you miss a functionality in the sparse tensor support, please submit a feature request to https://github.com/pytorch/pytorch/issues. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/aten/src/ATen/SparseCsrTensorImpl.cpp:51.)
  return torch.sparse_csr_tensor(indptr, in

four-stage done in 10.0s


## Stage-by-stage strain progression

Stage 1 (geometry only) gives a coarse floor; stage 2 (full distortion)
lowers it; stage 3's spline mops up the residual ΔR structure; stage 4
reports the honest test-set floor.


In [4]:
print(f'stage 1 (geom only)   : {res.stage1.final_strain_uE:8.2f} ue   '
      f'Lsd={float(res.stage1.unpacked["Lsd"]):.1f}')
print(f'stage 2 (full)        : {res.stage2.final_strain_uE:8.2f} ue   '
      f'Lsd={float(res.stage2.unpacked["Lsd"]):.1f}')
print(f'stage 3 spline train RMS (um): {res.stage3_train_rms_um:8.4f}')
print(f'stage 3 spline test  RMS (um): {res.stage3_test_rms_um:8.4f}  (honest)')
print(f'stage 4 strain (full mean)   : {res.stage4_strain_uE:8.2f} ue')
print(f'stage 4 strain (full median) : {res.stage4_strain_uE_med:8.2f} ue')
print(f'stage 4 strain (test mean)   : {res.stage4_strain_uE_test:8.2f} ue  (honest)')
print(f'stage 4 strain (test median) : {res.stage4_strain_uE_test_med:8.2f} ue  (honest)')


stage 1 (geom only)   :   374.01 ue   Lsd=1000131.2
stage 2 (full)        :   153.40 ue   Lsd=1000032.6
stage 3 spline train RMS (um):  27.3265
stage 3 spline test  RMS (um):  42.0526  (honest)
stage 4 strain (full mean)   :   153.66 ue
stage 4 strain (full median) :    38.62 ue
stage 4 strain (test mean)   :   245.57 ue  (honest)
stage 4 strain (test median) :    79.01 ue  (honest)


In [5]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

labels = ['stage1\n(geom)', 'stage2\n(full)', 'stage4\n(median)']
vals = [res.stage1.final_strain_uE, res.stage2.final_strain_uE,
        res.stage4_strain_uE_med]
fig, ax = plt.subplots(figsize=(5.5, 4))
ax.bar(labels, vals, color=['C0', 'C1', 'C2'])
ax.set_ylabel('mean pseudo-strain (ue)')
ax.set_title('Four-stage strain reduction (synthetic CeO2)')
for i, v in enumerate(vals):
    ax.text(i, v, f'{v:.0f}', ha='center', va='bottom')
ax.grid(axis='y', alpha=0.3)
fig.tight_layout()
fig.savefig('four_stage_strain.png', dpi=120)
plt.close(fig)
print('wrote four_stage_strain.png')


wrote four_stage_strain.png


## The stage-3 spline as a callable ΔR map

`res.stage3_spline_fn` is a callable `(Y_pix, Z_pix) -> ΔR (µm)`.  It is
the empirical residual-correction surface — the same object that gets
persisted as a v1-compatible binary for downstream integration.


In [6]:
fn = res.stage3_spline_fn
if fn is not None:
    Yq = np.array([200.0, 400.0, 600.0, 800.0])
    Zq = np.array([512.0, 512.0, 512.0, 512.0])
    dR = fn(Yq, Zq)
    print('spline ΔR (um) sampled along a vertical line through BC:')
    for y, d in zip(Yq, np.atleast_1d(dR)):
        print(f'  Y={y:6.0f}  Z=512  ΔR={float(d):+.3f} um')
else:
    print('stage-3 spline disabled')


spline ΔR (um) sampled along a vertical line through BC:
  Y=   200  Z=512  ΔR=-1.007 um
  Y=   400  Z=512  ΔR=-0.287 um
  Y=   600  Z=512  ΔR=-0.242 um
  Y=   800  Z=512  ΔR=+0.239 um


## On the per-ring δr_k

The four-stage workflow also supports **per-ring radial offsets**
`δr_k` (the F2 fix for per-ring DC structure; see notebook 05).  On a
*single* image δr_k is degenerate with a per-(hkl) lattice shift; the
spline of stage 3 captures the spatial (Y, Z) part of the residual
that the per-ring scalar cannot.  To break the δr_k ↔ Δa degeneracy
you need multiple distances — which is exactly **notebook 22**.

## Takeaway

`autocalibrate_four_stage` reproduces paper-3's published staged
workflow in one call, reporting an **honest held-out** strain floor
and a reusable TPS residual-correction map.
